# Implement a Transformer Decoder Block from Scratch
### Problem Statement
The Transformer Decoder Block is the core repeating unit in decoder-only LLMs (GPT, Llama, Mistral, etc.). Each block combines all the components you've built in this series:

```
x → RMSNorm → Multi-Head Attention → + (residual) → RMSNorm → SwiGLU FFN → + (residual) → output
```

This is the **Pre-Norm** architecture used by modern LLMs. A full model stacks N of these blocks (e.g., 32 for Llama 2 7B).

Your goal is to implement a complete decoder block using the components from previous exercises, with causal masking for autoregressive generation.

---

### Requirements

1. **Components**
   - RMSNorm (from `08-RMS-Norm`)
   - Multi-Head Attention (from `04-Multi-Head-Attention`) with **causal mask**
   - SwiGLU Feed-Forward Network (from `10-Feed-Forward-Network`)
   - Residual connections (from `09-Residual-Connection`)

2. **Causal Masking**
   - Generate a lower-triangular mask so each token can only attend to itself and previous tokens.
   - This enforces the autoregressive property.

3. **Architecture (Pre-Norm)**
   ```
   h = x + Attention(RMSNorm(x), causal_mask)
   output = h + FFN(RMSNorm(h))
   ```

4. **Validate**
   - Output shape must be `(batch_size, seq_len, d_model)`.
   - Causal mask must prevent future token leakage.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Pre-Norm architecture (norm before sublayer, not after).
- ✅ Causal masking must be correct.
- ✅ All sub-components should be implemented from scratch (no `nn.TransformerDecoderLayer`).

---

<details>
  <summary>💡 Hint</summary>

  - Causal mask: `torch.tril(torch.ones(seq_len, seq_len))` — lower triangular matrix of ones.
  - In the attention scores, mask positions with 0 should be filled with `-inf` before softmax.
  - The block has no learnable parameters of its own — it just orchestrates sub-modules.
  - Remember: Pre-Norm means `x + sublayer(norm(x))`, not `norm(x + sublayer(x))`.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Re-implement the building blocks (or import from previous notebooks)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.scale

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention with causal mask support."""
    def __init__(self, num_heads, d_model):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads

        # TODO: Create Q, K, V, and output projections
        ...

    def forward(self, x, mask=None):
        # TODO: Self-attention with optional causal mask
        # Note: this is SELF-attention, so q=k=v=x
        ...

In [ ]:
class SwiGLUFeedForward(nn.Module):
    """SwiGLU FFN."""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or int(2 / 3 * 4 * d_model)
        # TODO: Create W_gate, W_up, W_down
        ...

    def forward(self, x):
        # TODO: SwiGLU(x) = W_down(Swish(W_gate(x)) * W_up(x))
        ...

In [ ]:
class TransformerDecoderBlock(nn.Module):
    """
    A single Transformer Decoder Block (Pre-Norm architecture).
    x → RMSNorm → Attention → + residual → RMSNorm → FFN → + residual → output
    """
    def __init__(self, num_heads: int, d_model: int, d_ff: int = None):
        super().__init__()
        # TODO: Create attention, ffn, and two RMSNorm layers
        ...

    def forward(self, x, mask=None):
        """
        Args:
            x (Tensor): Input of shape (batch_size, seq_len, d_model)
            mask (Tensor, optional): Causal mask

        Returns:
            Tensor: Output of shape (batch_size, seq_len, d_model)
        """
        # TODO: Pre-Norm decoder block
        # 1. h = x + attention(norm1(x), mask)
        # 2. output = h + ffn(norm2(h))
        ...

In [ ]:
# Helper: create causal mask
def create_causal_mask(seq_len):
    """Returns a lower-triangular boolean mask of shape (1, 1, seq_len, seq_len)."""
    # TODO: Create causal mask
    ...

In [ ]:
# Test
torch.manual_seed(42)
batch_size, seq_len, d_model, num_heads = 2, 6, 32, 4

block = TransformerDecoderBlock(num_heads, d_model)
x = torch.randn(batch_size, seq_len, d_model)
mask = create_causal_mask(seq_len)

output = block(x, mask)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Mask shape:   {mask.shape}")
assert output.shape == x.shape

# Count parameters
total_params = sum(p.numel() for p in block.parameters())
print(f"\nTotal parameters: {total_params:,}")
print("\n✅ Transformer Decoder Block works correctly!")

In [ ]:
# Bonus: Stack multiple blocks to form a mini decoder
num_layers = 4
blocks = nn.ModuleList([TransformerDecoderBlock(num_heads, d_model) for _ in range(num_layers)])

h = x
for block in blocks:
    h = block(h, mask)

print(f"Input:  {x.shape}")
print(f"After {num_layers} blocks: {h.shape}")
total = sum(p.numel() for p in blocks.parameters())
print(f"Total params ({num_layers} layers): {total:,}")